In [3]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect("PerfectPetClub.db")

print("Database connection successful! yipee")

Database connection successful! yipee


In [54]:
import os
import sqlite3
import pandas as pd


def process_perfect_pet_club_data(db_path, output_csv_path):
    # 1. Connect to SQLite Database
    conn = sqlite3.connect(db_path)

    # 2. Extract Data with Explicit SQL Joins and Column Aliasing
    query = """
    SELECT 
        s.id AS sale_id,
        s.order_date,
        s.expected_date,
        s.delivered_date,
        
        c.id AS customer_id,
        c.name AS customer_name,
        c.email AS customer_email,
        c.county AS customer_county,
        c.postcode AS customer_postcode,
        
        sp.product_id,
        sp.quantity,
        
        p.name AS product_name,
        p.price AS unit_price,
        
        pc.id AS category_id,
        pc.name AS category_name,
        
        pay.payment_gateway,
        pay.amount AS payment_amount,
        
        sub.start_date AS sub_start_date,
        sub.end_date AS sub_end_date,
        sub.auto_renew AS sub_auto_renew,
        st.name AS subscription_plan,
        st.delivery_service
        
    FROM sale s
    JOIN customer c ON s.customer_id = c.id
    JOIN sale_product sp ON s.id = sp.sale_id
    JOIN product p ON sp.product_id = p.id
    LEFT JOIN product_category pc ON p.product_category_id = pc.id
    LEFT JOIN payment pay ON s.payment_id = pay.id
    LEFT JOIN subscription sub ON c.subscription_id = sub.id
    LEFT JOIN subscription_type st ON sub.subscription_type_id = st.id
    """

    df = pd.read_sql_query(query, conn)
    conn.close()

    # 3. Handle Category Imputation
    df["category_name"] = df["category_name"].fillna("Subscription")
    df["category_id"] = df["category_id"].fillna(-1).astype(int)

    # 4. Datetime Formatting
    date_cols = [
        "order_date",
        "expected_date",
        "delivered_date",
        "sub_start_date",
        "sub_end_date",
    ]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    # 5. Feature Engineering & Corrected Delivery Logic
   # ---------------------------------------------------------

    # A. Total Line Item Price
    df["total_line_item_price"] = df["quantity"] * df["unit_price"]

    # B. Identify Subscription Items
    is_subscription = df["product_name"].str.contains(
        "subscription", case=False, na=False
    )
    df["is_subscription_item"] = is_subscription.astype(int)

    # C. Delivery Binary Flag (is_delivered: 1 if delivered or subscription, 0 if pending/undelivered)
    df["is_delivered"] = 0
    # Physical products with a valid delivery date get 1
    df.loc[(~is_subscription) & (df["delivered_date"].notna()), "is_delivered"] = 1
    # Digital subscription items are always considered delivered (1)
    df.loc[is_subscription, "is_delivered"] = 1

    # D. Delivery Duration Days: Total shipping time (delivered_date - order_date)
    # Undelivered items get 0 duration days
    df["delivery_duration_days"] = 0
    df.loc[df["is_delivered"] == 1, "delivery_duration_days"] = (
        df["delivered_date"] - df["order_date"]
    ).dt.days

    # E. Impute Missing expected_date Dynamically for Physical Products
    avg_delivery_days = df.loc[
        (~is_subscription) & (df["is_delivered"] == 1), "delivery_duration_days"
    ].mean()
    fallback_days = (
        int(avg_delivery_days) if pd.notna(avg_delivery_days) else 3
    )

    missing_expected = (~is_subscription) & (df["expected_date"].isna())
    df.loc[missing_expected, "expected_date"] = df.loc[
        missing_expected, "order_date"
    ] + pd.Timedelta(days=fallback_days)

    # F. Delivery Delay Days: Delivered Date - Expected Date
    df["delivery_delay_days"] = (
        df["delivered_date"] - df["expected_date"]
    ).dt.days

    # Set delay days strictly to 0 for early/on-time deliveries AND undelivered items
    df.loc[
        (df["delivery_delay_days"] <= 0) | (df["is_delivered"] == 0),
        "delivery_delay_days",
    ] = 0

    # Reset duration and delay metrics for digital subscription services
    df.loc[is_subscription, "delivery_duration_days"] = 0
    df.loc[is_subscription, "delivery_delay_days"] = 0

    # G. Categorical Delivery Status
    df["delivery_status"] = "Delivered On Time"

    df.loc[is_subscription, "delivery_status"] = "Subscription Service"

    df.loc[
        (~is_subscription)
        & (df["is_delivered"] == 1)
        & (df["delivery_delay_days"] > 0),
        "delivery_status",
    ] = "Delayed"

    df.loc[
        (~is_subscription) & (df["is_delivered"] == 0), "delivery_status"
    ] = "Pending / Undelivered"

    # H. Churn Risk Signal: Undelivered Flag
    df["is_undelivered"] = (df["is_delivered"] == 0).astype(int)

    # I. Missing Payment Gateway Imputation
    df["payment_gateway"] = df["payment_gateway"].fillna("Unknown")

    # 6. Save Processed Master Dataset

# ---------------------------------------------------------
    # 6. Reorder Columns & Save Master Dataset
    # ---------------------------------------------------------
    ordered_cols = [
        "customer_id",
        "customer_name",
        "customer_email",
        "customer_county",
        "customer_postcode",
        "sale_id",
        "product_id",
        "quantity",
        "product_name",
         "unit_price", 
        "category_id",
        "category_name",
        "total_line_item_price",
        "payment_gateway",
        "payment_amount",
        "order_date",
        "expected_date",
        "delivered_date",
        "delivery_duration_days",
        "delivery_delay_days",
        "delivery_status",
        "is_delivered",
        "is_undelivered",
        "sub_start_date",
        "sub_end_date",
        "sub_auto_renew",
        "subscription_plan",
        "delivery_service",
        "is_subscription_item",
    ]

    # Reorder columns before saving
    df = df[ordered_cols]

    # Save to CSV
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df.to_csv(output_csv_path, index=False)

    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df.to_csv(output_csv_path, index=False)

    print(f"Data Processing Completed successfully!")
    print(
        f"Exported {len(df)} records to processed dataset path: '{output_csv_path}'"
    )

    return df






if __name__ == "__main__":
    BASE_DIR = os.path.dirname(os.path.abspath("deneme.ipynb"))
    DB_PATH = os.path.join(BASE_DIR, "PerfectPetClub.db")
    OUTPUT_PATH = os.path.join(BASE_DIR, "..", "data", "processed_sales.csv")

    print("Database Path:", DB_PATH)
    print("Output CSV Path:", OUTPUT_PATH)

    df = process_perfect_pet_club_data(
        db_path=DB_PATH, output_csv_path=OUTPUT_PATH
    )

Database Path: c:\Users\berkay\source\repos\customer-churn-prediction\notebooks\PerfectPetClub.db
Output CSV Path: c:\Users\berkay\source\repos\customer-churn-prediction\notebooks\..\data\processed_sales.csv
Data Processing Completed successfully!
Exported 59759 records to processed dataset path: 'c:\Users\berkay\source\repos\customer-churn-prediction\notebooks\..\data\processed_sales.csv'


In [41]:
import sqlite3
import os

# Veritabanı yolun
BASE_DIR = os.path.dirname(os.path.abspath("deneme.ipynb"))
DB_PATH = os.path.join(BASE_DIR, "PerfectPetClub.db")

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# 1. Kategori ekle (varsa atlar)
cursor.execute("INSERT OR IGNORE INTO product_category (id, name) VALUES (27, 'Subscription');")

# 2. Ürünlerin category_id'sini 27 yap
cursor.execute("""
UPDATE product 
SET product_category_id = 27 
WHERE name LIKE '%subscription%';
""")

# 3. İŞTE EN ÖNEMLİ KISIM: Veritabanına kalıcı yaz (COMMIT)
conn.commit()

# Kontrol sorgusu: Bakalım Python veritabanından ne okuyor?
cursor.execute("SELECT id, name, product_category_id FROM product WHERE name LIKE '%subscription%'")
updated_products = cursor.fetchall()

conn.close()

print("Veritabanından okunan güncel ürünler:")
for p in updated_products:
    print(p)

Veritabanından okunan güncel ürünler:
(1, 'Premium 1 month subscription', 27)
(2, 'Basic 1 month subscription', 27)
(3, 'SuperSaver 1 month subscription', 27)


In [34]:
df.head()

,sale_id,order_date,expected_date,delivered_date,customer_id,customer_name,customer_email,customer_county,customer_postcode,product_id,...,sub_start_date,sub_end_date,sub_auto_renew,subscription_plan,delivery_service,total_line_item_price,is_subscription_item,delivery_duration_days,delivery_delay_days,delivery_status
0,1,2026-01-01,NaT,NaT,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,3,...,2026-01-01,2026-01-31,0,SuperSaver,Economy72,299,1,-1.0,0.0,Subscription Service
1,2,2026-01-01,NaT,NaT,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,3,...,2026-01-01,2026-11-27,1,SuperSaver,Economy72,299,1,-1.0,0.0,Subscription Service
2,3,2026-01-01,NaT,NaT,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,1,...,2026-01-01,2026-11-27,1,Premium,SignedFor24,1199,1,-1.0,0.0,Subscription Service
3,4,2026-01-01,NaT,NaT,4,Omar Farooq,Omar.Farooq4@example.com,Kent,TN9 1LM,1,...,2026-01-01,2026-11-27,1,Premium,SignedFor24,1199,1,-1.0,0.0,Subscription Service
4,5,2026-01-01,NaT,NaT,5,Alejandro Blanco,Alejandro357@somemail.co.uk,Hertfordshire,WD7 8PQ,3,...,2026-01-01,2026-05-01,0,SuperSaver,Economy72,299,1,-1.0,0.0,Subscription Service


In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59759 entries, 0 to 59758
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_id             59759 non-null  int64         
 1   customer_name           59759 non-null  object        
 2   customer_email          59759 non-null  object        
 3   customer_county         59759 non-null  object        
 4   customer_postcode       59759 non-null  object        
 5   sale_id                 59759 non-null  int64         
 6   product_id              59759 non-null  int64         
 7   quantity                59759 non-null  int64         
 8   product_name            59759 non-null  object        
 9   unit_price              59759 non-null  int64         
 10  category_id             59759 non-null  int64         
 11  category_name           59759 non-null  object        
 12  total_line_item_price   59759 non-null  int64 

In [53]:
df.columns

Index(['customer_id', 'customer_name', 'customer_email', 'customer_county',
       'customer_postcode', 'sale_id', 'product_id', 'quantity',
       'product_name', 'unit_price', 'category_id', 'category_name',
       'total_line_item_price', 'payment_gateway', 'payment_amount',
       'order_date', 'expected_date', 'delivered_date',
       'delivery_duration_days', 'delivery_delay_days', 'delivery_status',
       'is_delivered', 'is_undelivered', 'sub_start_date', 'sub_end_date',
       'sub_auto_renew', 'subscription_plan', 'delivery_service',
       'is_subscription_item'],
      dtype='object')

In [49]:

subscription_sales = df[df["category_id"] == 27]
display(subscription_sales)

,sale_id,order_date,expected_date,delivered_date,customer_id,customer_name,customer_email,customer_county,customer_postcode,product_id,...,sub_auto_renew,subscription_plan,delivery_service,total_line_item_price,is_subscription_item,is_delivered,delivery_duration_days,delivery_delay_days,delivery_status,is_undelivered
0,1,2026-01-01 00:00:00.000,NaT,NaT,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,3,...,0,SuperSaver,Economy72,299,1,1,0.0,0.0,Subscription Service,0
1,2,2026-01-01 00:00:00.000,NaT,NaT,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,3,...,1,SuperSaver,Economy72,299,1,1,0.0,0.0,Subscription Service,0
2,3,2026-01-01 00:00:00.000,NaT,NaT,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,1,...,1,Premium,SignedFor24,1199,1,1,0.0,0.0,Subscription Service,0
3,4,2026-01-01 00:00:00.000,NaT,NaT,4,Omar Farooq,Omar.Farooq4@example.com,Kent,TN9 1LM,1,...,1,Premium,SignedFor24,1199,1,1,0.0,0.0,Subscription Service,0
4,5,2026-01-01 00:00:00.000,NaT,NaT,5,Alejandro Blanco,Alejandro357@somemail.co.uk,Hertfordshire,WD7 8PQ,3,...,0,SuperSaver,Economy72,299,1,1,0.0,0.0,Subscription Service,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59515,23080,2026-11-08 19:38:41.468,NaT,NaT,136,Khaled Al-Sabah,KAl-Sabah561@notrealemail.com,Somerset,BS21 6AB,1,...,1,Premium,SignedFor24,1199,1,1,0.0,0.0,Subscription Service,0
59546,23094,2026-11-08 20:52:01.472,NaT,NaT,77,Grace Johnson,Johnson490@notrealemail.com,Essex,CM9 5PQ,2,...,1,Basic,Tracked48,799,1,1,0.0,0.0,Subscription Service,0
59593,23117,2026-11-09 17:35:20.486,NaT,NaT,96,Jude Morris,tjy309@fakemail.co.uk,Kent,CT5 1PQ,3,...,1,SuperSaver,Economy72,299,1,1,0.0,0.0,Subscription Service,0
59675,23155,2026-11-09 20:04:40.494,NaT,NaT,117,Liam Watson,Liam-Watson993@fakemail.co.uk,Cornwall,PL25 5CD,3,...,1,SuperSaver,Economy72,299,1,1,0.0,0.0,Subscription Service,0


In [ ]:
"""
Updating subs collumn into db

INSERT INTO product_category (name) VALUES ('Subscription');
UPDATE product 
SET product_category_id = (SELECT id FROM product_category WHERE name = 'Subscription')
WHERE name LIKE '%subscription%';

"""
